# SPOOL — photon-efficient quantitative FLIM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wonsang7/SPOOL/blob/main/SPOOL_demo_FINAL.ipynb)

**SPOOL** (Spatially Pooled Optical Observation Likelihood) is a training-free Poisson inverse framework that models the microscope point-spread function (PSF) explicitly.

This demo:
1. simulates a few-photon FLIM acquisition with 2 ns and 4 ns emitters,
2. reconstructs the same photon counts with pixel-wise Poisson MLE and SPOOL,
3. compares emitter-center lifetime RMSE, and
4. renders a four-panel comparison figure directly in Colab.

> **Quick start:** choose a GPU runtime, then run all cells.
> **Recommended runtime:** T4 GPU (`Runtime → Change runtime type → T4 GPU`).


## 1. Setup

Clone the SPOOL repository, check out the requested revision, and install dependencies. The cell is safe to rerun in the same Colab session.


In [ ]:
import sys
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Wonsang7/SPOOL.git"
REPO_DIR = Path("/content/SPOOL_repo")
REPO_REF = "main"  #@param {type:"string"}

if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir():
    print(f"⚠ Removing incomplete checkout: {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

if not (REPO_DIR / ".git").is_dir():
    result = subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Could not clone the SPOOL repository.")
    print("✓ Cloned SPOOL repository")
else:
    print("✓ Using existing SPOOL checkout")

fetch = subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags"], text=True, capture_output=True)
if fetch.returncode != 0:
    print(fetch.stderr)
    raise RuntimeError("git fetch failed")

if REPO_REF == "main":
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "-q", "origin/main"], check=True)
else:
    checkout = subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", REPO_REF], text=True, capture_output=True)
    if checkout.returncode != 0:
        print(checkout.stderr)
        raise RuntimeError(f"Could not check out REPO_REF={REPO_REF!r}")

required_files = [REPO_DIR / "flim_sim_ncpca.py", REPO_DIR / "run_multiemitter_benchmark.py"]
missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError("The selected repository revision is missing required demo files: " + ", ".join(missing))

requirements = REPO_DIR / "requirements.txt"
if requirements.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)

commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"✓ Ready: {REPO_DIR} @ {REPO_REF} ({commit})")


In [ ]:
import sys
import time
from pathlib import Path

repo_path = str(Path("/content/SPOOL_repo").resolve())
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

import numpy as np
import matplotlib.pyplot as plt

import flim_sim_ncpca as fs
import run_multiemitter_benchmark as bm

print(f"✓ Compute backend: {bm.BACKEND}")
if bm.USE_TORCH:
    import torch
    if torch.cuda.is_available():
        print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("⚠ Torch is available but CUDA is not active.")
else:
    print("⚠ Running on CPU. The demo still works, but reconstruction will be slower.")


## 2. Model and reconstruction setup

The reconstruction uses a lifetime dictionary and an explicit PSF model. Pixel-wise MLE estimates lifetime independently at each detector pixel. SPOOL instead estimates source-space coefficients through a PSF-coupled likelihood so photons distributed over one diffraction-limited footprint constrain the same underlying source.


In [ ]:
bm.D_REC = bm.build_decay_basis(bm.TAU_REC)

D_d = bm.to_dev(bm.D_REC)
Dsum_d = bm.to_dev(bm.D_REC.sum(axis=1))

H = W = fs.IMAGE_SIZE
o = bm.make_otf(np.asarray(fs.PSF), (H, W))

if bm.USE_TORCH:
    import torch
    ones = torch.ones((bm.K_REC, H, W), dtype=torch.float64, device=bm.DEVICE)
    C = bm.conv_batch(ones, o) * Dsum_d.reshape(bm.K_REC, 1, 1)
else:
    ones = np.ones((bm.K_REC, H, W))
    C = bm.conv_batch(ones, o) * np.asarray(Dsum_d).reshape(bm.K_REC, 1, 1)

print(f"✓ Dictionary: K={bm.K_REC}, {bm.TAU_REC[0]:.1f}–{bm.TAU_REC[-1]:.1f} ns")
print(f"✓ PSF: {fs.PSF_FWHM_NM:.0f} nm FWHM ({fs.PSF_SIGMA_PX:.3f} px sigma at {fs.PIXEL_SIZE_NM} nm/px)")


## 3. Simulate one few-photon acquisition

The scene contains 15 emitters with 2 ns and 4 ns lifetimes mixed in the same field. The default photon level gives roughly 5 detected photons per foreground pixel.


In [ ]:
#@title Choose the photon budget
photons_per_bead = 200  #@param [50, 100, 200, 400, 800] {type:"raw"}

cfg = {"n_species0": None}
DEMO_SEED = 2000

bg = bm.background_for_level(photons_per_bead, cfg)
fs.BACKGROUND_RATE = bg

rng = np.random.default_rng(DEMO_SEED)
A_true, coords = fs.generate_scene(
    n_beads=bm.N_BEADS,
    photons_per_bead=photons_per_bead,
    rng=rng,
    n_species0=cfg["n_species0"],
)

species = np.array([np.argmax(A_true[:, y, x]) for (x, y) in coords])
tau_true = np.asarray(fs.TAUS_NS)[species]

Y, Lam = fs.simulate_measurement(A_true, rng)
fg_mean, _ = bm.photon_budget(Lam, coords)

print(f"✓ Simulated {len(coords)} emitters")
print(f"✓ Mean detected photons per foreground pixel: {fg_mean:.1f}")
print(f"✓ Background rate used by the benchmark: {bg:.4g}")


## 4. Reconstruct and display the comparison

The figure is rendered to PNG in memory and displayed explicitly, so it does not depend on Matplotlib's interactive backend.


In [ ]:
import io
from IPython.display import Image, display

t0 = time.time()
with bm.reconstruction_dictionary():
    Y_d = bm.to_dev(Y)
    A_mle = bm.to_np(bm.mle_gpu(Y_d, D_d, Dsum_d, bg))
    A_sp = bm.to_np(bm.joint_gpu(Y_d, D_d, o, C, bg))
elapsed = time.time() - t0

tau_mle = bm.lifetime_at(A_mle, coords, bm.TAU_REC)
tau_sp = bm.lifetime_at(A_sp, coords, bm.TAU_REC)
rmse_mle = float(np.sqrt(np.mean((tau_mle - tau_true) ** 2)))
rmse_sp = float(np.sqrt(np.mean((tau_sp - tau_true) ** 2)))
improvement = rmse_mle / rmse_sp if rmse_sp > 0 else np.inf

print(f"✓ Reconstruction time: {elapsed:.1f} s ({bm.BACKEND})")
print()
print("Emitter-center lifetime RMSE")
print(f"  Pixel-wise MLE : {rmse_mle:.3f} ns")
print(f"  SPOOL          : {rmse_sp:.3f} ns")
print(f"  RMSE improvement: {improvement:.2f}×")

cmap = plt.cm.rainbow.copy()
cmap.set_bad("black")
panels = [
    ("Ground truth", bm.masked_map(bm.tau_map(A_true, np.asarray(fs.TAUS_NS)), A_true.sum(0), frac=0.5)),
    ("Detected photons", None),
    ("Pixel-wise MLE", bm.masked_map(bm.tau_map(A_mle, bm.TAU_REC), A_mle.sum(0), frac=0.05)),
    ("SPOOL", bm.masked_map(bm.tau_map(A_sp, bm.TAU_REC), A_sp.sum(0), frac=0.05)),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, (title, lifetime_map) in zip(axes, panels):
    if lifetime_map is None:
        im = ax.imshow(Y.sum(axis=2), cmap="magma")
        fig.colorbar(im, ax=ax, fraction=0.046, label="photons")
    else:
        im = ax.imshow(lifetime_map, cmap=cmap, vmin=2, vmax=4)
        fig.colorbar(im, ax=ax, fraction=0.046, label="lifetime (ns)")
    ax.set_title(title)
    ax.axis("off")

fig.suptitle(f"Heterogeneous phantom — {fg_mean:.1f} photons per foreground pixel", y=1.03)
fig.tight_layout()

buf = io.BytesIO()
fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
plt.close(fig)
buf.seek(0)
display(Image(data=buf.getvalue()))


> **Key observation:** the pixel-wise reconstruction estimates lifetime separately across the diffraction-limited footprint, whereas SPOOL uses the spatial photon distribution through the PSF-coupled likelihood.


## 5. Optional: miniature photon-efficiency sweep

This quick sweep uses three photon levels × three Poisson trials. It is off by default so that `Run all` finishes after the core demonstration.


In [ ]:
#@title Run the miniature photon sweep?
RUN_PHOTON_SWEEP = False  #@param {type:"boolean"}

if not RUN_PHOTON_SWEEP:
    print("Mini sweep skipped. Enable the checkbox and rerun this cell to run it.")
else:
    levels, n_trials = [50, 200, 800], 3
    sweep = {"MLE": [], "SPOOL": []}
    fg_x = []
    background_before_sweep = fs.BACKGROUND_RATE
    try:
        for pl in levels:
            bg_sweep = bm.background_for_level(pl, cfg)
            fs.BACKGROUND_RATE = bg_sweep
            r_m, r_s, f_ = [], [], []
            for trial in range(n_trials):
                rng = np.random.default_rng(DEMO_SEED + trial)
                A_t, crd = fs.generate_scene(n_beads=bm.N_BEADS, photons_per_bead=pl, rng=rng, n_species0=cfg["n_species0"])
                sp = np.array([np.argmax(A_t[:, y, x]) for (x, y) in crd])
                t_t = np.asarray(fs.TAUS_NS)[sp]
                Yt, Lt = fs.simulate_measurement(A_t, rng)
                f_.append(bm.photon_budget(Lt, crd)[0])
                with bm.reconstruction_dictionary():
                    Yd = bm.to_dev(Yt)
                    Am = bm.to_np(bm.mle_gpu(Yd, D_d, Dsum_d, bg_sweep))
                    Aj = bm.to_np(bm.joint_gpu(Yd, D_d, o, C, bg_sweep))
                r_m.append(np.sqrt(np.mean((bm.lifetime_at(Am, crd, bm.TAU_REC) - t_t) ** 2)))
                r_s.append(np.sqrt(np.mean((bm.lifetime_at(Aj, crd, bm.TAU_REC) - t_t) ** 2)))
            sweep["MLE"].append((np.mean(r_m), np.std(r_m)))
            sweep["SPOOL"].append((np.mean(r_s), np.std(r_s)))
            fg_x.append(np.mean(f_))
            print(f"{fg_x[-1]:5.1f} photons/px | MLE {np.mean(r_m):.3f} ns | SPOOL {np.mean(r_s):.3f} ns")
    finally:
        fs.BACKGROUND_RATE = background_before_sweep

    fig, ax = plt.subplots(figsize=(5.5, 4))
    for name, color in [("MLE", "gray"), ("SPOOL", "crimson")]:
        mu = [v[0] for v in sweep[name]]
        sd = [v[1] for v in sweep[name]]
        ax.errorbar(fg_x, mu, yerr=sd, marker="o", color=color, label=name, capsize=3)
    ax.set_xscale("log")
    ax.set_xlabel("mean photons per foreground pixel")
    ax.set_ylabel("emitter-center lifetime RMSE (ns)")
    ax.set_title("Heterogeneous phantom — mini sweep")
    ax.legend()
    fig.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


## 6. Going further

- **Full manuscript benchmark:** `python run_multiemitter_benchmark.py`
- **Experimental analyses:** `run_dual_labeled_flim.py` and `run_hyperspectral.py` require the corresponding photon-count datasets.
- **Reproducibility:** the setup cell prints the exact Git commit used in the session.
